In [0]:
import json, os, sys
from datetime import datetime
from urllib.parse import quote
from zoneinfo import ZoneInfo
sys.path.append(os.path.abspath('..'))

from src.config import SECRET_SCOPE, TICKERS_BRAPI
from src.extrator import coletar_json, finalizar
from src.http_client import criar_sessao

dbutils.widgets.dropdown('modo', 'intradiario', ['intradiario', 'diario', 'backfill'])
modo = dbutils.widgets.get('modo')

BASE = 'https://brapi.dev/api'
sessao = criar_sessao()
headers = {'Authorization': f"Bearer {dbutils.secrets.get(SECRET_SCOPE, 'brapi_token')}"}
logs = []


def em_pregao():
    agora = datetime.now(ZoneInfo('America/Sao_Paulo'))
    return agora.weekday() < 5 and (10, 0) <= (agora.hour, agora.minute) <= (18, 30)


if modo == 'intradiario':
    if not em_pregao():
        dbutils.notebook.exit('Fora do pregão: coleta intradiária da brapi ignorada para poupar cota')
    for ticker in TICKERS_BRAPI:
        url = f"{BASE}/quote/{quote(ticker, safe='')}"
        logs.append(coletar_json(sessao, 'brapi', 'cotacoes', url, headers))

if modo in ('diario', 'backfill'):
    janela = '5d' if modo == 'diario' else '3mo'
    for ticker in TICKERS_BRAPI:
        url = f"{BASE}/quote/{quote(ticker, safe='')}"
        logs.append(coletar_json(sessao, 'brapi', 'historico', url, headers,
                                 params={'range': janela, 'interval': '1d'}))

    pagina = 1
    while pagina <= 50:  # trava de segurança contra paginação infinita
        log = coletar_json(sessao, 'brapi', 'catalogo_ativos', f'{BASE}/quote/list', headers,
                           params={'page': pagina, 'limit': 100})
        logs.append(log)
        if not log['sucesso']:
            break
        with open(log['arquivo_raw'], encoding='utf-8') as f:
            resposta = json.loads(json.loads(f.read())['payload'])
        if not resposta.get('hasNextPage'):
            break
        pagina += 1

finalizar(logs)